# Comparaison Ovide / Bibles, par theme

Complement a `comparaison_ovide_bibles.ipynb` (nuage UMAP unique, toutes illustrations
melangees) : celui-ci separe la comparaison par theme partage, pour repondre plus
directement a la question de depart -- comparer les illustrations d'Ovide a celles des
Bibles **dans les themes**.

Deux vues, plus concretes que le nuage global :

1. **Petits multiples par theme** -- une mini-projection UMAP par theme partage
   (creation du monde, creation de l'homme/Eve, deluge), chacune ne contenant QUE les
   illustrations Bible et Ovide de ce theme precis. Le nuage global melangeait ces 3 themes
   avec 222 illustrations Bible "autre theme biblique" (Babel, Tentation...) sans equivalent
   Ovide -- plus de la moitie des points, qui n'apportaient rien a la comparaison par theme
   et genaient la lecture.
2. **Galerie des meilleures correspondances** -- pour chaque illustration Ovide, sa Bible la
   plus proche (n'importe quel theme, meme calcul que dans le notebook global), mais
   affichee comme une paire d'images cote a cote plutot que comme un lien entre deux points
   a survoler : comparaison visuelle directe.

**Prerequis** : `mini_rag_iconographique/vector_base.ipynb` deja execute au moins une fois
(genere `data/vector_bases/bibles_siglip.pkl` et `ovide_bnu_corpus_comparaison_siglip.pkl`).
`lib/d3.v7.min.js` a cote de ce notebook (gitignore, deja present ici -- copie de celui de
`comparaison_ovide_bibles/lib/`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

RACINE = Path("../../").resolve()
DOSSIER_VECTOR_DB = RACINE / "data" / "vector_bases"
DOSSIER_SORTIE = RACINE / "resultats" / "comparaison_ovide_bibles"
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
CHEMIN_SORTIE = DOSSIER_SORTIE / "comparaison_par_theme.html"

## 1. Chargement et taxonomie commune

Meme regroupement que dans `comparaison_ovide_bibles.ipynb` (12 classes fines des Bibles ->
3 themes partages avec Ovide + "autre_bible" pour le reste) -- verifie ci-dessous qu'il ne
reste aucune illustration non mappee.

In [ ]:
base_bibles = pd.read_pickle(DOSSIER_VECTOR_DB / "bibles_siglip.pkl")
base_ovide = pd.read_pickle(DOSSIER_VECTOR_DB / "ovide_bnu_corpus_comparaison_siglip.pkl")

MAPPING_THEME_BIBLE = {
    "Creation du MONDE": "creation_monde",
    "Creation HOMME": "creation_homme",
    "Creation EVE": "creation_homme",
    "Creation HORS ENTRAINEMENT": "creation_homme",
    "Deluge": "deluge",
    "Deluge AVANT": "deluge",
    "Deluge APRES": "deluge",
    "Babel": "autre_bible",
    "Cain et Abel": "autre_bible",
    "Chasses du paradis": "autre_bible",
    "Sacrifice ABRAHAM": "autre_bible",
    "Tentation": "autre_bible",
}

base_bibles = base_bibles.copy()
base_bibles["source"] = "bible"
base_bibles["theme_groupe"] = base_bibles["theme"].map(MAPPING_THEME_BIBLE)
base_bibles["theme_detail"] = base_bibles["theme"]
base_bibles["titre"] = np.nan

non_mappes = base_bibles["theme_groupe"].isna().sum()
if non_mappes:
    valeurs = sorted(base_bibles.loc[base_bibles["theme_groupe"].isna(), "theme"].unique())
    print(f"ATTENTION : {non_mappes} illustrations Bible non mappees a un theme_groupe : {valeurs}")
else:
    print(f"Toutes les {len(base_bibles)} illustrations Bible sont mappees a un theme_groupe.")

base_ovide = base_ovide.copy()
base_ovide["source"] = "ovide"
base_ovide["theme_groupe"] = base_ovide["theme"]
base_ovide["theme_detail"] = base_ovide["theme"]

THEMES_PARTAGES = ["creation_monde", "creation_homme", "deluge"]

print()
print("Bible  :", dict(base_bibles.groupby("theme_groupe").size()))
print("Ovide  :", dict(base_ovide.groupby("theme_groupe").size()))

## 2. Petits multiples : une projection UMAP par theme partage

Pour chaque theme, une projection UMAP independante calculee **uniquement** sur les
illustrations Bible + Ovide de ce theme (pas les 222 "autre theme biblique", pas les deux
autres themes partages) : la question posee est locale a chaque theme ("dans le deluge, les
deux corpus se ressemblent-ils ?"), une projection dediee y repond plus directement qu'un
sous-ensemble lu dans le nuage global.

Pour chaque illustration, on calcule aussi sa meilleure similarite croisee (source opposee)
**au sein du theme** -- utile en infobulle pour voir immediatement si un point a un vrai
equivalent proche de l'autre corpus, ou s'il est isole. Une verification *trustworthiness*
(comme dans le notebook global) confirme que chaque petite projection reste fidele a
l'espace 768D d'origine malgre le faible nombre de points.

In [3]:
import umap
from sklearn.manifold import trustworthiness

COLONNES_POINT = ["chemin", "source", "theme_groupe", "theme_detail", "titre", "embedding", "url_page", "url_image"]

points_par_theme = {}
for theme in THEMES_PARTAGES:
    sous = pd.concat([
        base_bibles.loc[base_bibles["theme_groupe"] == theme, COLONNES_POINT],
        base_ovide.loc[base_ovide["theme_groupe"] == theme, COLONNES_POINT],
    ], ignore_index=True)

    X = np.array(sous["embedding"].tolist())
    n_voisins = min(15, len(sous) - 1)
    reducteur = umap.UMAP(n_neighbors=n_voisins, min_dist=0.3, metric="cosine", random_state=42)
    coords = reducteur.fit_transform(X)
    sous["x"], sous["y"] = coords[:, 0], coords[:, 1]

    # Meilleure similarite vers l'AUTRE source, dans ce theme uniquement.
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, -1)
    est_bible = (sous["source"] == "bible").to_numpy()
    sous["meilleure_sim_croisee"] = [
        float(sim[i, np.where(est_bible != est_bible[i])[0]].max())
        for i in range(len(sous))
    ]

    k_conf = min(5, len(sous) - 1)
    confiance = trustworthiness(X, coords, n_neighbors=k_conf, metric="cosine")
    points_par_theme[theme] = {"points": sous, "trustworthiness": confiance}

    n_bible = int((sous["source"] == "bible").sum())
    n_ovide = int((sous["source"] == "ovide").sum())
    print(f"{theme:16s} : {n_bible:3d} Bible + {n_ovide:3d} Ovide = {len(sous):3d} points, trustworthiness={confiance:.3f}")

/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


creation_monde   :  29 Bible +  25 Ovide =  54 points, trustworthiness=0.844
creation_homme   :  56 Bible +  17 Ovide =  73 points, trustworthiness=0.903
deluge           :  65 Bible +  31 Ovide =  96 points, trustworthiness=0.942


/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 3. Galerie des meilleures correspondances (Ovide -> Bible)

Meme calcul que la section "Correspondances Ovide -> Bible" du notebook global (meilleure
Bible pour chaque Ovide, n'importe quel theme), mais exporte ici comme des **paires
d'images**, triees par similarite decroissante -- pour une comparaison visuelle directe,
sans avoir a survoler des points dans un nuage.

In [4]:
X_ovide_full = np.array(base_ovide["embedding"].tolist())
X_bible_full = np.array(base_bibles["embedding"].tolist())
sim_croisee_totale = cosine_similarity(X_ovide_full, X_bible_full)

paires = []
for i in range(len(base_ovide)):
    j = int(sim_croisee_totale[i].argmax())
    ligne_o = base_ovide.iloc[i]
    ligne_b = base_bibles.iloc[j]
    paires.append({
        "similarite": float(sim_croisee_totale[i, j]),
        "memeTheme": bool(ligne_o["theme_groupe"] == ligne_b["theme_groupe"]),
        "ovide": {
            "img": ligne_o["url_image"] if pd.notna(ligne_o["url_image"]) else "",
            "page": ligne_o["url_page"] if pd.notna(ligne_o["url_page"]) else "",
            "titre": str(ligne_o["titre"])[:90] if pd.notna(ligne_o.get("titre")) else "",
            "theme": ligne_o["theme_groupe"],
        },
        "bible": {
            "img": ligne_b["url_image"] if pd.notna(ligne_b["url_image"]) else "",
            "page": ligne_b["url_page"] if pd.notna(ligne_b["url_page"]) else "",
            "theme": ligne_b["theme_groupe"],
            "detail": ligne_b["theme_detail"],
        },
    })

paires.sort(key=lambda p: -p["similarite"])

n_meme_theme = sum(p["memeTheme"] for p in paires)
print(f"{len(paires)} paires (une par illustration Ovide, meilleure correspondance Bible toutes classes confondues)")
print(f"{n_meme_theme} / {len(paires)} partagent le meme theme cote Bible ({n_meme_theme / len(paires):.0%})")

73 paires (une par illustration Ovide, meilleure correspondance Bible toutes classes confondues)
26 / 73 partagent le meme theme cote Bible (36%)


## 4. Export HTML (petits multiples + galerie)

In [5]:
import json as jsonlib

def noeud_json(row):
    return {
        "x": float(row["x"]), "y": float(row["y"]),
        "source": row["source"], "detail": row["theme_detail"],
        "titre": (str(row["titre"])[:90] if pd.notna(row.get("titre")) else ""),
        "img": row["url_image"] if pd.notna(row["url_image"]) else "",
        "page": row["url_page"] if pd.notna(row["url_page"]) else "",
        "simCroisee": row["meilleure_sim_croisee"],
    }

THEMES_JSON = {
    theme: {
        "noeuds": [noeud_json(r) for _, r in d["points"].iterrows()],
        "trustworthiness": d["trustworthiness"],
    }
    for theme, d in points_par_theme.items()
}

DONNEES_JSON = jsonlib.dumps({"themes": THEMES_JSON, "paires": paires}, ensure_ascii=False)
print(f"JSON pret : {len(DONNEES_JSON) / 1024:.0f} Ko")

JSON pret : 116 Ko


In [6]:
LABEL_THEME = {
    "creation_monde": "Creation du monde",
    "creation_homme": "Creation de l'homme / Eve",
    "deluge": "Deluge",
}

boutons_filtre_theme = "".join(
    f'<button data-groupe="theme" data-valeur="{theme}">{label}</button>'
    for theme, label in LABEL_THEME.items()
)

panneaux_themes = "".join(
    f'''<div class="panneau-theme">
    <h3>{LABEL_THEME[theme]}</h3>
    <p class="note-panneau">{len([n for n in THEMES_JSON[theme]["noeuds"] if n["source"] == "bible"])} Bible
      + {len([n for n in THEMES_JSON[theme]["noeuds"] if n["source"] == "ovide"])} Ovide
      &middot; fidelite de la projection : {THEMES_JSON[theme]["trustworthiness"]:.2f}</p>
    <div class="cadre-mini"><svg data-theme="{theme}"></svg></div>
  </div>'''
    for theme in THEMES_PARTAGES
)

HTML_TEMPLATE = r"""<!doctype html>
<html lang="fr">
<head>
<meta charset="utf-8">
<title>Comparaison Ovide / Bibles, par theme</title>
<style>
  .viz-root {
    color-scheme: light;
    --surface-1:      #fcfcfb;
    --page-plane:     #f9f9f7;
    --text-primary:   #0b0b0b;
    --text-secondary: #52514e;
    --text-muted:     #898781;
    --gridline:       #e1e0d9;
    --border:         rgba(11,11,11,0.10);
    --c-bible: #2a78d6;
    --c-ovide: #d9832b;
  }
  @media (prefers-color-scheme: dark) {
    :root:where(:not([data-theme="light"])) .viz-root {
      color-scheme: dark;
      --surface-1:      #1a1a19;
      --page-plane:     #0d0d0d;
      --text-primary:   #ffffff;
      --text-secondary: #c3c2b7;
      --text-muted:     #898781;
      --gridline:       #2c2c2a;
      --border:         rgba(255,255,255,0.10);
      --c-bible: #3987e5;
      --c-ovide: #e8a052;
    }
  }
  :root[data-theme="dark"] .viz-root {
    color-scheme: dark;
    --surface-1:      #1a1a19;
    --page-plane:     #0d0d0d;
    --text-primary:   #ffffff;
    --text-secondary: #c3c2b7;
    --text-muted:     #898781;
    --gridline:       #2c2c2a;
    --border:         rgba(255,255,255,0.10);
    --c-bible: #3987e5;
    --c-ovide: #e8a052;
  }
  * { box-sizing: border-box; }
  html, body { margin: 0; padding: 0; overflow-x: hidden; }
  body { font-family: system-ui, -apple-system, "Segoe UI", sans-serif; background: var(--page-plane); color: var(--text-primary); }
  .viz-root { padding: 20px; max-width: 1200px; margin: 0 auto; position: relative; }
  h1 { font-size: 1.15em; margin: 0 0 4px; }
  h2 { font-size: 1em; margin: 28px 0 4px; }
  .sous-titre, .note { color: var(--text-secondary); font-size: 0.85em; margin: 0 0 16px; }

  .legende-globale { display: flex; gap: 16px; align-items: center; font-size: 0.8em; color: var(--text-secondary); margin: 0 0 14px; }
  .legende-globale .item { display: flex; align-items: center; gap: 6px; }
  .legende-globale svg { width: 12px; height: 12px; }

  .rangee-themes { display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 16px; }
  .panneau-theme { background: var(--surface-1); border: 1px solid var(--border); border-radius: 8px; padding: 12px; }
  .panneau-theme h3 { margin: 0 0 2px; font-size: 0.95em; }
  .note-panneau { color: var(--text-muted); font-size: 0.75em; margin: 0 0 8px; }
  .cadre-mini { position: relative; width: 100%; aspect-ratio: 1 / 1; border: 1px solid var(--gridline); border-radius: 6px; overflow: hidden; }
  .cadre-mini svg { width: 100%; height: 100%; display: block; }
  .marque { stroke: var(--border); stroke-width: 1px; cursor: pointer; }

  .infobulle {
    position: absolute; pointer-events: none;
    background: var(--surface-1); border: 1px solid var(--border); border-radius: 6px;
    padding: 8px; font-size: 0.78em; color: var(--text-primary);
    box-shadow: 0 4px 16px rgba(0,0,0,0.18); max-width: 260px; opacity: 0; transition: opacity 0.1s; z-index: 10;
  }
  .infobulle img { width: 100%; border-radius: 4px; margin-bottom: 6px; display: block; }
  .infobulle .theme { color: var(--text-secondary); }

  .filtres-galerie { display: flex; flex-wrap: wrap; align-items: center; gap: 14px; margin: 0 0 14px; }
  .groupe-filtres { display: flex; gap: 6px; flex-wrap: wrap; align-items: center; }
  .groupe-filtres .etiquette { font-size: 0.78em; color: var(--text-muted); margin-right: 2px; }
  .filtres-galerie button {
    font: inherit; font-size: 0.8em; border: 1px solid var(--border); background: transparent;
    color: var(--text-secondary); padding: 5px 11px; border-radius: 14px; cursor: pointer;
  }
  .filtres-galerie button.actif { background: var(--text-primary); color: var(--page-plane); border-color: var(--text-primary); }

  .grille-galerie { display: grid; grid-template-columns: repeat(auto-fill, minmax(280px, 1fr)); gap: 14px; }
  .carte-paire { background: var(--surface-1); border: 1px solid var(--border); border-radius: 8px; padding: 10px; }
  .carte-paire.masquee { display: none; }
  .paire-images { display: flex; align-items: center; gap: 6px; cursor: zoom-in; }
  .paire-images figure { margin: 0; flex: 1; min-width: 0; }
  .paire-images img { width: 100%; height: 110px; object-fit: cover; border-radius: 4px; display: block; background: var(--gridline); transition: opacity .2s; }
  .paire-images img:not([src]) { opacity: 0; }
  .paire-images img.erreur { visibility: hidden; }
  .paire-images .fleche { color: var(--text-muted); font-size: 1.1em; flex: none; }
  figcaption { font-size: 0.72em; color: var(--text-secondary); margin-top: 4px; }
  figcaption b.bible { color: var(--c-bible); }
  figcaption b.ovide { color: var(--c-ovide); }
  .paire-info { display: flex; justify-content: space-between; align-items: center; margin-top: 8px; font-size: 0.75em; color: var(--text-muted); }
  .paire-info .sim { font-variant-numeric: tabular-nums; }
  .paire-info .autre-theme { color: var(--c-ovide); }
  .paire-info a { color: var(--text-secondary); text-decoration: none; border-bottom: 1px dotted var(--text-muted); }

  .lightbox {
    position: fixed; inset: 0; background: rgba(8,7,6,.88); display: none;
    align-items: center; justify-content: center; z-index: 3000; cursor: zoom-out; padding: 30px;
  }
  .lightbox.visible { display: flex; }
  .lightbox-paire { display: flex; gap: 22px; max-width: 95vw; max-height: 88vh; cursor: default; }
  .lightbox-paire figure { margin: 0; display: flex; flex-direction: column; align-items: center; min-width: 0; }
  .lightbox-paire img {
    max-width: 44vw; max-height: 78vh; object-fit: contain; border-radius: 6px; display: block;
    box-shadow: 0 8px 40px rgba(0,0,0,.5); background: var(--surface-1);
  }
  .lightbox-paire img.erreur { display: none; }
  .lightbox-paire .img-manquant {
    display: none; width: 300px; padding: 40px 0; text-align: center; font-size: 0.85em;
    color: #ccc; background: rgba(255,255,255,.08); border-radius: 6px;
  }
  .lightbox-paire img.erreur + .img-manquant { display: block; }
  .lightbox-paire figcaption { color: #ece8e0; font-size: 0.85em; margin-top: 10px; text-align: center; max-width: 44vw; }
  .lightbox-paire figcaption b.bible { color: #6fb0ff; }
  .lightbox-paire figcaption b.ovide { color: #ffb35c; }
  .fermer-lightbox {
    position: absolute; top: 18px; right: 26px; background: none; border: none;
    color: #fff; font-size: 34px; line-height: 1; cursor: pointer; z-index: 3001;
  }
</style>
</head>
<body>
<div class="viz-root">
  <h1>Comparaison Ovide / Bibles, par theme</h1>
  <p class="sous-titre">Complement au nuage UMAP global (`comparaison_ovide_bibles.ipynb`) : ici, une projection
    dediee par theme partage, puis la galerie des meilleures correspondances Ovide -&gt; Bible.</p>

  <div class="legende-globale">
    <span class="item"><svg viewBox="-5 -5 10 10"><circle r="4" fill="var(--c-bible)"/></svg> Bible</span>
    <span class="item"><svg viewBox="-6 -5 12 11"><path d="M0,-5 L5,5 L-5,5 Z" fill="var(--c-ovide)"/></svg> Ovide</span>
  </div>

  <div class="rangee-themes">
    __PANNEAUX_THEMES__
  </div>

  <h2>Galerie des meilleures correspondances (Ovide -&gt; Bible)</h2>
  <p class="note">__N_PAIRES__ illustrations Ovide, chacune avec sa Bible la plus proche (n'importe quel theme,
    embeddings SigLIP 768D) -- __N_MEME_THEME__ / __N_PAIRES__ partagent le meme theme cote Bible. Triees par
    similarite decroissante. Images chargees progressivement (Gallica limite le nombre de requetes rapprochees).
    Cliquer une paire pour l'agrandir.</p>
  <div class="filtres-galerie" id="filtres-galerie">
    <div class="groupe-filtres">
      <span class="etiquette">Theme :</span>
      <button data-groupe="theme" data-valeur="tous" class="actif">Tous</button>
      __BOUTONS_FILTRE_THEME__
    </div>
    <div class="groupe-filtres">
      <span class="etiquette">Correspondance :</span>
      <button data-groupe="correspondance" data-valeur="tous" class="actif">Tous</button>
      <button data-groupe="correspondance" data-valeur="meme">Meme theme</button>
      <button data-groupe="correspondance" data-valeur="autre">Autre theme</button>
    </div>
  </div>
  <div class="grille-galerie" id="grille-galerie"></div>
</div>

<div class="lightbox" id="lightbox">
  <button class="fermer-lightbox" id="fermer-lightbox" aria-label="Fermer">&times;</button>
  <div class="lightbox-paire" id="lightbox-paire"></div>
</div>

<script>__D3_JS__</script>
<script>
const DONNEES = __DONNEES_JSON__;
const LABEL_THEME = __LABEL_THEME_JSON__;

// Les URLs d'image (Gallica / MDZ) pointent par defaut vers la pleine resolution IIIF
// ("/full/full/0/..."), inutilement lourde pour des vignettes de 110px. On demande une
// taille reduite via le parametre IIIF "size" (",280" = hauteur 280px, largeur
// proportionnelle) pour les vignettes de la grille.
function redimensionner(url, taille) {
  return url.replace("/full/full/0/", `/full/,${taille}/0/`);
}
const miniature = url => redimensionner(url, 280);
// MDZ (digitale-sammlungen.de) : demander une grande taille personnalisee (ex. ",1100")
// declenche une redirection cassee cote serveur vers une image plus petite que la vignette
// elle-meme (observe en pratique -- l'image ne s'affiche jamais, malgre une reponse 200).
// Gallica gere ces tailles sans probleme. Pour la vue agrandie (un seul chargement a la
// fois, pas de souci de volume), on garde donc la reduction pour Gallica et la pleine
// resolution pour MDZ.
function grandeImage(url) {
  return url.includes("gallica.bnf.fr") ? redimensionner(url, 1100) : url;
}

function ligneImage(url, classe) {
  return `<img src="${miniature(url)}" loading="lazy" class="${classe || ''}" onerror="this.classList.add('erreur')">`;
}

// --- Petits multiples : une mini-projection D3 par theme -------------------
const formeParSource = {
  bible: d3.symbol().type(d3.symbolCircle).size(46),
  ovide: d3.symbol().type(d3.symbolTriangle).size(46),
};

function dessinerPanneau(svgEl, noeuds) {
  const svg = d3.select(svgEl);
  const cadre = svgEl.closest(".cadre-mini");
  const largeur = cadre.clientWidth, hauteur = cadre.clientHeight;
  svg.attr("viewBox", [0, 0, largeur, hauteur]);
  const g = svg.append("g");

  const marge = 18;
  const xScale = d3.scaleLinear().domain(d3.extent(noeuds, d => d.x)).range([marge, largeur - marge]);
  const yScale = d3.scaleLinear().domain(d3.extent(noeuds, d => d.y)).range([hauteur - marge, marge]);

  const infobulle = d3.select("body").append("div").attr("class", "infobulle");

  g.selectAll("path")
    .data(noeuds)
    .join("path")
    .attr("class", "marque")
    .attr("transform", d => `translate(${xScale(d.x)},${yScale(d.y)})`)
    .attr("d", d => formeParSource[d.source]())
    .attr("fill", d => d.source === "bible" ? "var(--c-bible)" : "var(--c-ovide)")
    .on("mouseenter", (evt, d) => {
      const rectCadre = cadre.getBoundingClientRect();
      infobulle
        .style("opacity", 1)
        .html(`
          ${ligneImage(d.img, "img-vignette")}
          <div><b>${d.source === "bible" ? "Bible" : "Ovide"}</b> &mdash; ${d.detail}</div>
          ${d.titre ? `<div class="theme">${d.titre}</div>` : ""}
          <div class="theme">Meilleure correspondance croisee (ce theme) : ${d.simCroisee.toFixed(3)}</div>
        `)
        .style("left", (rectCadre.left + window.scrollX + xScale(d.x) + 14) + "px")
        .style("top", (rectCadre.top + window.scrollY + yScale(d.y) + 10) + "px");
    })
    .on("mouseleave", () => infobulle.style("opacity", 0));
}

document.querySelectorAll(".cadre-mini svg").forEach(svgEl => {
  const theme = svgEl.dataset.theme;
  dessinerPanneau(svgEl, DONNEES.themes[theme].noeuds);
});

// --- Galerie des meilleures correspondances ---------------------------------
// La galerie affiche jusqu'a 146 images (73 paires) hebergees chez Gallica/MDZ. Les charger
// toutes d'un coup (meme en taille reduite) declenche un 429 (rate limit) chez Gallica dans
// environ 40% des cas -- verifie en pratique. On charge donc en file, quelques images a la
// fois (CONCURRENCE_MAX), avec une reprise a delai croissant en cas d'echec, plutot que de
// laisser le navigateur tirer toutes les requetes en parallele (comportement par defaut de
// `loading="lazy"` des que plusieurs dizaines d'images entrent dans le viewport d'un coup,
// ex. lors d'un defilement rapide).
const CONCURRENCE_MAX = 3;
let enCoursDeChargement = 0;
const fileAttente = [];

function chargerUneImage(img) {
  return new Promise(resolve => {
    function essayer(tentativesRestantes, delaiMs) {
      img.onload = () => resolve(true);
      img.onerror = () => {
        if (tentativesRestantes > 0) {
          setTimeout(() => essayer(tentativesRestantes - 1, delaiMs * 2), delaiMs);
        } else {
          img.classList.add("erreur");
          resolve(false);
        }
      };
      img.src = img.dataset.src;
    }
    essayer(3, 1500);
  });
}

function traiterFileAttente() {
  while (enCoursDeChargement < CONCURRENCE_MAX && fileAttente.length) {
    const img = fileAttente.shift();
    enCoursDeChargement++;
    chargerUneImage(img).finally(() => { enCoursDeChargement--; traiterFileAttente(); });
  }
}

const observateurImages = new IntersectionObserver((entrees, obs) => {
  entrees.forEach(entree => {
    if (entree.isIntersecting) {
      fileAttente.push(entree.target);
      obs.unobserve(entree.target);
      traiterFileAttente();
    }
  });
}, {rootMargin: "300px"});

function ligneImageDifferee(url, classe) {
  return `<img data-src="${miniature(url)}" class="${classe || ''}">`;
}

const grille = document.getElementById("grille-galerie");
grille.innerHTML = DONNEES.paires.map((p, i) => `
  <div class="carte-paire" data-theme="${p.ovide.theme}" data-meme-theme="${p.memeTheme ? 'oui' : 'non'}">
    <div class="paire-images" onclick="ouvrirComparaison(${i})">
      <figure>
        ${ligneImageDifferee(p.ovide.img)}
        <figcaption><b class="ovide">Ovide</b> &mdash; ${LABEL_THEME[p.ovide.theme] || p.ovide.theme}</figcaption>
      </figure>
      <div class="fleche">&#8596;</div>
      <figure>
        ${ligneImageDifferee(p.bible.img)}
        <figcaption><b class="bible">Bible</b> &mdash; ${p.bible.detail}</figcaption>
      </figure>
    </div>
    <div class="paire-info">
      <span class="sim">similarite ${p.similarite.toFixed(3)}${p.memeTheme ? "" : ' &middot; <span class="autre-theme">autre theme</span>'}</span>
      <span>${p.ovide.page ? `<a href="${p.ovide.page}" target="_blank" rel="noopener">Ovide</a>` : ""} ${p.bible.page ? `<a href="${p.bible.page}" target="_blank" rel="noopener">Bible</a>` : ""}</span>
    </div>
  </div>
`).join("");

document.querySelectorAll(".grille-galerie img[data-src]").forEach(img => observateurImages.observe(img));

// --- Filtres (theme x correspondance meme/autre theme, combines en ET) ------
let filtreTheme = "tous";
let filtreCorrespondance = "tous";

function appliquerFiltresGalerie() {
  document.querySelectorAll(".carte-paire").forEach(carte => {
    const okTheme = filtreTheme === "tous" || carte.dataset.theme === filtreTheme;
    const okCorrespondance = filtreCorrespondance === "tous"
      || (filtreCorrespondance === "meme" && carte.dataset.memeTheme === "oui")
      || (filtreCorrespondance === "autre" && carte.dataset.memeTheme === "non");
    carte.classList.toggle("masquee", !(okTheme && okCorrespondance));
  });
}

document.getElementById("filtres-galerie").addEventListener("click", evt => {
  const bouton = evt.target.closest("button");
  if (!bouton) return;
  const groupe = bouton.dataset.groupe;
  document.querySelectorAll(`#filtres-galerie button[data-groupe="${groupe}"]`).forEach(b => b.classList.toggle("actif", b === bouton));
  if (groupe === "theme") filtreTheme = bouton.dataset.valeur;
  else filtreCorrespondance = bouton.dataset.valeur;
  appliquerFiltresGalerie();
});

// --- Agrandissement d'une paire (clic sur les deux vignettes) ---------------
const lightbox = document.getElementById("lightbox");
const lightboxPaire = document.getElementById("lightbox-paire");

function ligneImageGrande(url) {
  return `<img src="${grandeImage(url)}" alt="" onerror="this.classList.add('erreur')">
    <div class="img-manquant">Image indisponible</div>`;
}

function ouvrirComparaison(i) {
  const p = DONNEES.paires[i];
  lightboxPaire.innerHTML = `
    <figure>
      ${ligneImageGrande(p.ovide.img)}
      <figcaption><b class="ovide">Ovide</b> &mdash; ${LABEL_THEME[p.ovide.theme] || p.ovide.theme}${p.ovide.titre ? " &mdash; " + p.ovide.titre : ""}</figcaption>
    </figure>
    <figure>
      ${ligneImageGrande(p.bible.img)}
      <figcaption><b class="bible">Bible</b> &mdash; ${p.bible.detail}</figcaption>
    </figure>
  `;
  lightbox.classList.add("visible");
}

function fermerLightbox() {
  lightbox.classList.remove("visible");
  lightboxPaire.innerHTML = "";
}

document.getElementById("fermer-lightbox").addEventListener("click", fermerLightbox);
lightbox.addEventListener("click", evt => { if (evt.target === lightbox) fermerLightbox(); });
document.addEventListener("keydown", evt => { if (evt.key === "Escape") fermerLightbox(); });
</script>
</body>
</html>"""

D3_JS = (Path(".") / "lib" / "d3.v7.min.js").read_text(encoding="utf-8")

html_final = (
    HTML_TEMPLATE
    .replace("__D3_JS__", D3_JS)
    .replace("__DONNEES_JSON__", DONNEES_JSON)
    .replace("__LABEL_THEME_JSON__", jsonlib.dumps(LABEL_THEME, ensure_ascii=False))
    .replace("__PANNEAUX_THEMES__", panneaux_themes)
    .replace("__BOUTONS_FILTRE_THEME__", boutons_filtre_theme)
    .replace("__N_PAIRES__", str(len(paires)))
    .replace("__N_MEME_THEME__", str(n_meme_theme))
)

CHEMIN_SORTIE.write_text(html_final, encoding="utf-8")
print(f"Visualisation generee : {CHEMIN_SORTIE}")
print(f"Taille : {CHEMIN_SORTIE.stat().st_size / 1024:.0f} Ko")

Visualisation generee : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/comparaison_ovide_bibles/comparaison_par_theme.html
Taille : 407 Ko
